In [ ]:
# ==========================================================
# セル1：ライブラリと共通モジュール fft_common の読み込み・解析設定
# ==========================================================
# 【対象】Futaba形式（1行目が "Time:" で始まるCSV）
#   もう一方の形式は 05_NR500複数全ファイル用_圧力経時変化フーリエ変換.ipynb を使ってください。
#
# ★加工対象CSVには一切書き込まない（読み取り専用）★
# ==========================================================
import os
import sys
import time
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

# --- 共通モジュール fft_common.py を探して読み込む ---
# ノートブックの実行時カレントディレクトリが一定しないため、
# 想定される場所を順に探す。見つからなければ理由を明示して止める。
_here = os.getcwd()
_cands = [
    _here,
    os.path.join(_here, "コードフォルダ"),
    os.path.dirname(_here),
    os.path.join(os.path.dirname(_here), "コードフォルダ"),
]
for _c in _cands:
    if os.path.isfile(os.path.join(_c, "fft_common.py")):
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break
else:
    raise FileNotFoundError(
        "fft_common.py が見つかりません。\n探した場所:\n  "
        + "\n  ".join(_cands)
        + "\nコードフォルダと同じ場所（またはその親フォルダ）で実行してください。"
    )

import fft_common
from fft_common import (
    VOLT_TO_MPA, amp_spectrum, detect_format, folder_tag, list_csv_by_format,
    metrics, read_any, setup_japanese_font,
)

setup_japanese_font(plt)
warnings.filterwarnings("ignore", category=RuntimeWarning,
                        message="All-NaN slice encountered")

# ==========================================================
# ⚙️ 解析設定（ここを変えると解析方法が変わります）
# ==========================================================
TARGET_FORMAT = "futaba"
CHANNELS = ["CH03", "CH04"]

# --- ゼロ点合わせ（先頭 N 行の平均を全行から引く）---
ZERO_ADJUST = False      # Futabaは元からMPaで基準線も0近傍のため既定OFF
ZERO_ROWS = 1000                    # ゼロ点合わせに使う先頭行数

# --- スペクトルの求め方 ---
USE_EVENT_WINDOW = True   # True : 圧力が立ち上がっているイベント区間だけを解析（推奨）
                          # False: 記録全体を解析（＝従来の動作。無信号区間のノイズが混入）
DETREND = True            # FFT前に解析区間の平均を引く（False が従来の動作）
WINDOW = "hann"           # "hann"（漏れが少ない）/ "none"（＝従来の矩形窓）
# ==========================================================
# ※グラフの見た目（周波数軸の範囲・対数軸など）は セル4の先頭 にあります

print("✅ 準備完了")
print(f"   共通モジュール : {fft_common.__file__}")
print(f"   対象形式       : {TARGET_FORMAT}   チャンネル: {', '.join(CHANNELS)}")
if ZERO_ADJUST:
    print(f"   ゼロ点合わせ   : ON（先頭 {ZERO_ROWS} 行の平均を全行から引く）")
else:
    print("   ゼロ点合わせ   : OFF")
print(f"   解析区間       : {'イベント区間のみ' if USE_EVENT_WINDOW else '記録全体（従来動作）'}")
print(f"   窓関数         : {WINDOW} / 直流除去: {DETREND}")
print("👉 次のセルを実行してください。")

In [ ]:
# ==========================================================
# セル2：メインフォルダの選択
# ==========================================================
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
main_dir = filedialog.askdirectory(title="解析対象のメインフォルダを選択してください")
root.destroy()

if not main_dir:
    print("⚠️ フォルダ選択がキャンセルされました。次のセルには進まず、やり直してください。")
else:
    print(f"✅ 選択されたメインフォルダ:\n{main_dir}")

In [ ]:
# ==========================================================
# セル3：対象ファイルの列挙と形式の絞り込み
# ==========================================================
if not main_dir:
    raise ValueError("メインフォルダが選択されていません。セル2を再実行してください。")

print("🔍 CSVを走査中...（1行目を見て形式を判定します）")
buckets = list_csv_by_format(main_dir)
target_files = buckets[TARGET_FORMAT]
others = sum(([p] for k, v in buckets.items() if k != TARGET_FORMAT for p in v), [])
n_all = sum(len(v) for v in buckets.values())

print(f"\n見つかったCSVファイル: 合計 {n_all} 件")
print(f"   ├ Futaba形式 : {len(target_files):5d} 件 ← 処理します")
for k, label in (("futaba", "Futaba形式"), ("nr500", "NR-500形式"), ("other", "対象外")):
    if k != TARGET_FORMAT and buckets[k]:
        print(f"   ├ {label:12s} : {len(buckets[k]):5d} 件 ← 除外します")

if target_files:
    print("\n【処理対象ファイルの例】")
    for f in target_files[:3]:
        print(" -", f)
    if len(target_files) > 3:
        print(f"   ... (他 {len(target_files) - 3} 件)")
else:
    print("\n⚠️ Futaba形式のCSVが1件も見つかりませんでした。")

# 除外したファイルも必ず見せる（無言でスキップしない）
if others:
    print("\n【除外したファイルの例】")
    for f in others[:5]:
        print(f" - {os.path.basename(f)}  ({detect_format(f) or '対象外'})")
    if len(others) > 5:
        print(f"   ... (他 {len(others) - 5} 件)")
    print("   → もう一方の形式は 05_NR500複数全ファイル用_圧力経時変化フーリエ変換.ipynb で処理できます。")

if target_files:
    print(f"\n✅ {len(target_files)} 件を処理します。次のセルへ進んでください。")

In [ ]:
# ==========================================================
# セル4：1ファイル1枚のFFTグラフを一括生成し、指標をCSVに出力
# ==========================================================
# ---- グラフの見た目 ----
LOG_Y = True                       # Y軸を対数にする（False が従来のリニア軸）
X_MIN, X_MAX, X_STEP = 0, 150, 10  # 周波数軸の範囲と目盛り間隔 [Hz]
Y_MIN, Y_MAX = 1e-4, None          # 対数軸のときの下限／上限（None で自動）
# ------------------------

if not target_files:
    raise ValueError("処理対象ファイルがありません。セル3を確認してください。")

output_dir_path = os.path.join(os.getcwd(), "fft_results")
os.makedirs(output_dir_path, exist_ok=True)

print("🚀 フーリエ変換を開始します...")
print(f"📁 保存先: {output_dir_path}")
print(f"⏱️ 目安: 約 {len(target_files) * 0.3 / 60:.1f} 分"
      f"（{len(target_files)} ファイル）\n")

records = []
errors = []
t0 = time.time()

for i, path in enumerate(target_files, 1):
    csv_file = os.path.basename(path)
    try:
        df, dt, info = read_any(path, channels=CHANNELS,
                                zero=ZERO_ADJUST, zero_rows=ZERO_ROWS)

        fig, ax = plt.subplots(figsize=(10, 5))
        for ch in CHANNELS:
            y = df[ch].to_numpy(dtype=np.float64)
            m, _spec, (i0, i1) = metrics(
                y, dt, use_event=USE_EVENT_WINDOW, detrend=DETREND, window=WINDOW,
                zero_rows=ZERO_ROWS if info["zero_applied"] else None)
            f, a = amp_spectrum(y[i0:i1 + 1], dt, detrend=DETREND, window=WINDOW)
            sel = (f >= X_MIN) & (f <= X_MAX)      # ★描画前に帯域を切る
            ax.plot(f[sel], a[sel], label=ch, alpha=0.85, linewidth=1.0)

            rec = {"file": csv_file, "path": path,
                   "group": folder_tag(os.path.dirname(path), main_dir),
                   "format": info["format"], "channel": ch,
                   "zero_offset_applied_MPa": info["zero_offset"].get(ch, 0.0)}
            rec.update(m)
            records.append(rec)

        ax.set_xlabel("Frequency [Hz]")
        ax.set_ylabel("Amplitude [MPa]")
        ax.set_title(f"FFT Spectrum － {csv_file}")
        ax.set_xlim(X_MIN, X_MAX)
        ax.set_xticks(np.arange(X_MIN, X_MAX + X_STEP, X_STEP))
        if LOG_Y:
            ax.set_yscale("log")
            ax.set_ylim(Y_MIN, Y_MAX)
        ax.legend()
        ax.grid(True, which="both", linestyle="--", alpha=0.5)
        fig.tight_layout()

        # 末端フォルダ名だけだと板厚違いが衝突するため、相対パス全体をタグにする
        tag = folder_tag(os.path.dirname(path), main_dir)
        fig.savefig(os.path.join(
            output_dir_path,
            f"[{tag}]_{os.path.splitext(csv_file)[0]}_fft.png"), dpi=150)
        plt.close(fig)

    except Exception as e:
        plt.close("all")
        errors.append((path, f"{type(e).__name__}: {e}"))

    if i % 25 == 0 or i == len(target_files):
        el = time.time() - t0
        print(f"  {i}/{len(target_files)} 件（経過 {el:.0f}秒 / "
              f"残り約 {el / i * (len(target_files) - i):.0f}秒）")

met = pd.DataFrame(records)
csv_path = os.path.join(output_dir_path, "metrics_per_file.csv")
met.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("\n" + "=" * 52)
print(f"🏁 完了！  画像 {len(target_files) - len(errors)} 枚 / "
      f"指標 {len(met)} 行")
print(f"💾 {csv_path}")
print("=" * 52)

# ---- 品質フラグの集計（無言でスキップしない）----
if len(met):
    print("\n【品質フラグの内訳】")
    for flag, cnt in met["flags"].value_counts().items():
        print(f"  {'✅' if flag == 'ok' else '⚠️'} {flag:32s} {cnt:5d} 件")
    bad = met[met["flags"] != "ok"]
    if len(bad):
        print(f"\n  ⚠️ 要確認 {len(bad)} 件（先頭10件。全件はCSVを参照）:")
        for _, r in bad.head(10).iterrows():
            print(f"     {r['file']} [{r['channel']}] → {r['flags']}")

if errors:
    print(f"\n❌ 読み込み失敗 {len(errors)} 件:")
    for p, msg in errors[:10]:
        print(f"   {os.path.basename(p)}: {msg}")

print("\n💡 カットオフ周波数を全ファイル横断で決めたい場合は")
print("   07_カットオフ周波数決定_全ファイル横断.ipynb を使ってください。")